# Diabetes Health Indicators + Social Media Analysis

This notebook brings together two streams:

1. Diabetes classification from health and lifestyle indicators
2. Social media text analysis to understand discussions related to diabetes, symptoms, lifestyle, and support

This is a practical example of how a health dataset and a public online discussion dataset can be studied together in a responsible and useful way.

Important note: the project uses a health dataset and a synthetic social-media-style dataset for demonstration. Real social media data must be collected ethically and legally, and personal identifiers must be removed.

## 1. Setup and imports


In [ ]:
import os
import urllib.request
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, classification_report, confusion_matrix
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression as TextLogisticRegression
from sklearn.model_selection import cross_val_score

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', context='notebook')


## 2. Load the diabetes dataset

We use the CDC Diabetes Health Indicators dataset from the reference project.

In [ ]:
DATA_PATH = 'data/diabetes_binary_health_indicators_BRFSS2015.csv'
DATA_URL = 'https://raw.githubusercontent.com/eviekenna/FinalProjectST558/main/diabetes_binary_health_indicators_BRFSS2015.csv'

if not os.path.exists(DATA_PATH):
    os.makedirs('data', exist_ok=True)
    urllib.request.urlretrieve(DATA_URL, DATA_PATH)

diabetes_df = pd.read_csv(DATA_PATH)
print(diabetes_df.shape)
display(diabetes_df.head())

## 3. Diabetes EDA: data quality and target distribution


In [ ]:
print('Missing values:
', diabetes_df.isna().sum().sum())
print('Duplicate rows:', diabetes_df.duplicated().sum())

diabetes_df['Diabetes_binary'] = diabetes_df['Diabetes_binary'].map({0: 'No Diabetes', 1: 'Prediabetes/Diabetes'})
target_counts = diabetes_df['Diabetes_binary'].value_counts()
print(target_counts)

sns.countplot(data=diabetes_df, x='Diabetes_binary', palette='Set2')
plt.title('Target Class Distribution')
plt.xlabel('Diabetes status')
plt.ylabel('Count')
plt.show()

majority_baseline = target_counts.max() / target_counts.sum()
print(f'Majority-class baseline accuracy: {majority_baseline:.3f}')

### Interpretation

The target is imbalanced. Therefore, a model with high accuracy does not automatically mean good diabetes detection. This is why recall and F1 or ROC-AUC matter.

## 4. Key health patterns


In [ ]:
risk_features = ['HighBP', 'HighChol', 'Smoker', 'PhysActivity', 'DiffWalk', 'Stroke']
for feature in risk_features:
    table = pd.crosstab(diabetes_df[feature], diabetes_df['Diabetes_binary'])
    print(f'\n=== {feature} ===')
    print((table.div(table.sum(axis=1), axis=0) * 100).round(2))

plt.figure(figsize=(12, 6))
sns.boxplot(data=diabetes_df, x='Diabetes_binary', y='BMI', palette='Set2')
plt.title('BMI by Diabetes Status')
plt.xlabel('Diabetes status')
plt.ylabel('BMI')
plt.show()

## 5. Build the diabetes classification model

We create a precise classification pipeline with preprocessing and class balancing.

In [ ]:
X = diabetes_df.drop(columns='Diabetes_binary')
y = diabetes_df['Diabetes_binary'].map({'No Diabetes': 0, 'Prediabetes/Diabetes': 1})

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

num_cols = X_train.select_dtypes(include='number').columns.tolist()
cat_cols = X_train.select_dtypes(exclude='number').columns.tolist()

numeric_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])
categorical_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])
preprocessor = ColumnTransformer([
    ('numeric', numeric_transformer, num_cols),
    ('categorical', categorical_transformer, cat_cols)
])

log_reg = Pipeline([
    ('preprocessor', preprocessor),
    ('model', LogisticRegression(max_iter=2000, class_weight='balanced', random_state=42))
])

rf_model = Pipeline([
    ('preprocessor', preprocessor),
    ('model', RandomForestClassifier(n_estimators=300, class_weight='balanced', random_state=42, n_jobs=-1))
])

models = {'Logistic Regression': log_reg, 'Random Forest': rf_model}

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]
    print(f'\n=== {name} ===')
    print('Accuracy:', round(accuracy_score(y_test, y_pred), 3))
    print('Precision:', round(precision_score(y_test, y_pred, zero_division=0), 3))
    print('Recall:', round(recall_score(y_test, y_pred, zero_division=0), 3))
    print('F1:', round(f1_score(y_test, y_pred, zero_division=0), 3))
    print('ROC AUC:', round(roc_auc_score(y_test, y_proba), 3))
    print(classification_report(y_test, y_pred, target_names=['No Diabetes', 'Prediabetes/Diabetes'], zero_division=0))

## 6. Social media text data: synthetic example

We now create a social-media-like dataset for demonstration. In a real project, this should come from publicly available posts collected legally, with all personal identifiers removed. The goal here is to show how NLP could be integrated with the health dataset.

In [ ]:
social_posts = [
    ('I have been feeling weak and thirsty all week. My blood sugar seems high.', 'diabetes_related', 'negative'),
    ('Feeling tired but trying to manage my diet and walk every day.', 'diabetes_related', 'neutral'),
    ('I am learning about prediabetes and trying to avoid sugary drinks.', 'diabetes_related', 'neutral'),
    ('My doctor told me I need to monitor my glucose after meals.', 'diabetes_related', 'negative'),
    ('I am excited to improve my sleep and exercise habits to lower my risk.', 'diabetes_related', 'positive'),
    ('Lunch tasted great but I should avoid processed snacks.', 'diabetes_related', 'neutral'),
    ('I love this healthy recipe for vegetables and quinoa.', 'diabetes_related', 'positive'),
    ('This weekend I am going hiking with friends to stay active.', 'diabetes_related', 'positive'),
    ('The weather is nice and I am enjoying a walk in the park.', 'not_diabetes_related', 'positive'),
    ('My favorite band just released a new song and I am listening all day.', 'not_diabetes_related', 'positive'),
    ('I need to finish my work assignment before the deadline.', 'not_diabetes_related', 'neutral'),
    ('Traffic was terrible on the way home today.', 'not_diabetes_related', 'negative'),
    ('I am worried about my blood sugar after eating fast food.', 'diabetes_related', 'negative'),
    ('Checking my steps and cutting back on soda is helping.', 'diabetes_related', 'positive'),
    ('The new fitness app is motivating me to stay consistent.', 'diabetes_related', 'positive'),
    ('My family is trying to eat healthier this month.', 'not_diabetes_related', 'neutral'),
    ('I am frustrated because my glucose is still high despite trying.', 'diabetes_related', 'negative'),
    ('The gym was full today but I still had a good workout.', 'diabetes_related', 'positive'),
    ('I have no energy today and the weather is gloomy.', 'not_diabetes_related', 'negative'),
    ('I am trying to stay calm and manage stress better.', 'not_diabetes_related', 'neutral')
]

social_df = pd.DataFrame(social_posts, columns=['text', 'topic_label', 'sentiment_label'])
display(social_df.head(10))

## 7. Social media text classification: diabetes-related vs non-diabetes-related

We transform text into numeric features using TF-IDF and classify diabetes-related posts using a simple logistic model.

In [ ]:
X_text = social_df['text']
y_text = social_df['topic_label'].map({'diabetes_related': 1, 'not_diabetes_related': 0})

X_text_train, X_text_test, y_text_train, y_text_test = train_test_split(
    X_text, y_text, test_size=0.3, random_state=42, stratify=y_text
)

text_pipe = Pipeline([
    ('tfidf', TfidfVectorizer(ngram_range=(1, 2), min_df=1)),
    ('model', TextLogisticRegression(max_iter=1000, random_state=42))
])

text_pipe.fit(X_text_train, y_text_train)
text_pred = text_pipe.predict(X_text_test)
print('Text model accuracy:', round(accuracy_score(y_text_test, text_pred), 3))
print(classification_report(y_text_test, text_pred, target_names=['Not Diabetes Related', 'Diabetes Related']))

## 8. Social media sentiment analysis

We can estimate sentiment using a simple dictionary-based approach that counts positive and negative words. This demonstrates a lightweight and explainable NLP workflow.

In [ ]:
positive_words = {'love', 'good', 'happy', 'excited', 'healthy', 'support', 'walking', 'exercise', 'improve', 'motivated', 'positive', 'great', 'helping'}
negative_words = {'worried', 'tired', 'frustrated', 'bad', 'high', 'weak', 'pain', 'stress', 'low', 'trouble', 'negative', 'sick'}

def simple_sentiment_score(text):
    tokens = text.lower().split()
    pos = sum(1 for t in tokens if t in positive_words)
    neg = sum(1 for t in tokens if t in negative_words)
    return pos - neg

social_df['sentiment_score'] = social_df['text'].apply(simple_sentiment_score)
social_df['sentiment_label_est'] = social_df['sentiment_score'].apply(lambda x: 'positive' if x > 0 else ('negative' if x < 0 else 'neutral'))

print(social_df[['text', 'sentiment_score', 'sentiment_label_est']].head(10))

social_summary = social_df.groupby('topic_label')['sentiment_score'].mean().reset_index()
print(social_summary)

sns.barplot(data=social_summary, x='topic_label', y='sentiment_score', palette='Set2')
plt.title('Average sentiment score by content type')
plt.xlabel('Content type')
plt.ylabel('Mean sentiment score')
plt.show()

## 9. Integrated interpretation: health + social media

This is the key idea: even though the health dataset and the social-media text dataset are different data sources, they can be connected conceptually.

The health model tells us which variables are strongly associated with diabetes. The social media analysis tells us which themes are discussed publicly by people who mention diabetes, risk, diet, exercise, stress, or symptoms.

In [ ]:
print('Health findings:')
print('- Diabetes prevalence is imbalanced and should not be judged by accuracy alone.')
print('- BMI, blood pressure, and cholesterol are strong health risk indicators.')
print('- Social media discussion often centers on symptoms, diet, exercise, and stress management.')
print('- Public concerns align with known risk factors and lifestyle interventions.')

# Example aggregated comparison
compare_df = pd.DataFrame({
    'Source': ['Health indicators', 'Social media text'],
    'Main signal': ['BMI, BP, cholesterol, age', 'Diet, symptoms, exercise, stress, support'],
    'Focus': ['Risk classification', 'Public discussion and sentiment']
})
display(compare_df)

## 10. Final conclusion

The project shows how learning can be extended from structured health data to social media analysis. A useful real-world workflow is:

1. Build a diabetes classification model from structured health features.
2. Analyze diabetes-related public discussions using NLP.
3. Compare the themes and find areas where public attention matches clinical risk patterns.
4. Interpret the findings with caution. Health behaviors and online discussion are not the same as an individual's medical status.

This is a sensible project design for applied machine learning, public health communications, and sentiment analysis.